In [ ]:
%pip install torch torchvision scikit-learn pandas pillow tqdm

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# Your existing dataset path
DATA_DIR = Path("../data/CUB_200_2011/CUB_200_2011")

# Check the dataset
assert DATA_DIR.exists(), f"Dataset not found: {DATA_DIR.resolve()}"

# Load image ID -> image path
image_paths = {}

with open(DATA_DIR / "images.txt", "r") as f:
    for line in f:
        image_id, image_path = line.strip().split(maxsplit=1)
        image_paths[int(image_id)] = image_path

# Load image ID -> class ID
image_classes = pd.read_csv(
    DATA_DIR / "image_class_labels.txt",
    sep=r"\s+",
    header=None,
    names=["image_id", "class_id"]
)

# Load attribute labels
# Use the existing dataframe if it is already in your notebook.
if "attribute_labels" not in globals():
    attribute_labels = pd.read_csv(
        DATA_DIR / "attributes" / "image_attribute_labels.txt",
        sep=r"\s+",
        header=None,
        usecols=[0, 1, 2],
        names=["image_id", "attribute_id", "is_present"],
        engine="python"
    )

# Create one row per image and one column per attribute
X_attr = attribute_labels.pivot_table(
    index="image_id",
    columns="attribute_id",
    values="is_present",
    aggfunc="first"
)

# Ensure all 312 attributes exist and are ordered correctly
X_attr = X_attr.reindex(
    index=sorted(image_paths.keys()),
    columns=range(1, 313),
    fill_value=0
).fillna(0).astype("float32")

print("Total images:", len(image_paths))
print("Attribute matrix:", X_attr.shape)
print("Unique classes:", image_classes["class_id"].nunique())
print("Missing attribute values:", int(X_attr.isna().sum().sum()))

# Display one sample image
sample_id = next(iter(image_paths))
sample_path = DATA_DIR / "images" / image_paths[sample_id]

img = Image.open(sample_path).convert("RGB")
plt.figure(figsize=(5, 4))
plt.imshow(img)
plt.title(f"Image ID: {sample_id}")
plt.axis("off")
plt.show()